# News Recommendation Model

This notebook builds the content-based TF-IDF and cosine-distance recommendation artifacts used by the Streamlit app. Run it from the project root after importing the CSV into MySQL.

In [16]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'dataset' / 'news.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_PATH = PROJECT_ROOT / 'dataset' / 'news.csv'
MODELS_PATH = PROJECT_ROOT / 'models'
MODELS_PATH.mkdir(exist_ok=True)
DATASET_PATH

PosixPath('/home/sahil/Desktop/AI-ML/Projects/Ideas/News Recommendation System - Ref/dataset/news.csv')

## Load and inspect the dataset

In [17]:
df = pd.read_csv(DATASET_PATH, low_memory=False)
print(f'Rows: {len(df):,}')
print('Columns:', ', '.join(df.columns))
display(df.head(2))

Rows: 54,328
Columns: article_id, category, headline, abstract, snippet, lead_paragraph, keywords, section, subsection, publication_date, source, byline, article_type, url, image_url, fetched_at


,article_id,category,headline,abstract,snippet,lead_paragraph,keywords,section,subsection,publication_date,source,byline,article_type,url,image_url,fetched_at
0,nyt://article/76559f92-28ec-5bfe-b55e-cc4a1975...,Business,Donald Trump Jr.’s Firm Leads $1 Billion Fundi...,The new round values the prediction market at ...,NaN,NaN,"[{""name"": ""Subject"", ""value"": ""Venture Capital...",Business,NaN,2026-09-01T00:00:45Z,The New York Times,By Lauren McCarthy,Article,https://www.nytimes.com/2026/08/31/business/po...,NaN,2026-09-08T05:46:54.996676+00:00
1,nyt://article/f84e05f0-e850-5120-bb17-2d07cd82...,Books,"Wendell Berry, Writer Who Extolled America’s A...","A Kentucky farmer, he railed against agribusin...",NaN,NaN,"[{""name"": ""Person"", ""value"": ""Berry, Wendell"",...",U.S.,NaN,2026-09-01T00:07:13Z,The New York Times,By Robert D. McFadden,Article,https://www.nytimes.com/2026/08/31/us/wendell-...,NaN,2026-09-08T05:46:54.996676+00:00


## Clean text and combine useful fields

The source has no full article body. Keywords are stored as JSON, so their values are extracted before joining them with the available news text.

In [18]:
def keywords_text(value):
    if pd.isna(value) or not str(value).strip():
        return ''
    try:
        parsed = json.loads(value)
        return ' '.join(item.get('value', '') for item in parsed if isinstance(item, dict))
    except (TypeError, ValueError, json.JSONDecodeError):
        return str(value)

TEXT_COLUMNS = ['headline', 'abstract', 'snippet', 'lead_paragraph', 'section', 'category']
for column in TEXT_COLUMNS:
    if column not in df:
        df[column] = ''
    df[column] = df[column].fillna('').astype(str)
df['keywords_text'] = df['keywords'].map(keywords_text) if 'keywords' in df else ''
df['combined_text'] = df[TEXT_COLUMNS].agg(' '.join, axis=1) + ' ' + df['keywords_text']
df['combined_text'] = df['combined_text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df[['headline', 'combined_text']].head(2)

,headline,combined_text
0,Donald Trump Jr.’s Firm Leads $1 Billion Fundi...,Donald Trump Jr.’s Firm Leads $1 Billion Fundi...
1,"Wendell Berry, Writer Who Extolled America’s A...","Wendell Berry, Writer Who Extolled America’s A..."


## TF-IDF vectorization and cosine model

In [19]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=100000,
    ngram_range=(1, 2),
    min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(df['combined_text'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

TF-IDF matrix shape: (54328, 100000)


## Test recommendations

In [20]:
def recommend(index, count=5):
    scores = cosine_similarity(tfidf_matrix[index], tfidf_matrix).ravel()
    scores[index] = -1
    selected = scores.argsort()[-count:][::-1]
    result = df.iloc[selected][['article_id', 'category', 'headline', 'abstract']].copy()
    result['similarity'] = scores[selected]
    return result

recommend(0)

,article_id,category,headline,abstract,similarity
51711,nyt://article/52a153e6-0cc6-5705-ab06-22a48562...,Politics,Donald Trump Jr. Is Joining a Venture Capital ...,"The firm, 1789 Capital, invests in products an...",0.478520
7229,nyt://article/d3532292-ada8-5c66-b4a9-5f918586...,Politics,"U.S. Begins Investigating Polymarket, a Test o...","Last year, the Commodity Futures Trading Commi...",0.448620
12154,nyt://article/32dbfe63-2eaa-5e5c-a0fa-8401ec99...,Technology,Trump Says He Dislikes Prediction Markets. His...,The White House has warned staff not to wager ...,0.360547
4109,nyt://article/669affa9-bb0b-5203-84b9-887c689d...,Business,Donald Trump Jr.’s Investment Firm Posts Stagg...,"The fledgling firm, run by the president’s old...",0.360251
25403,nyt://article/e125539a-0d4f-55e6-a8fc-78930c7c...,Politics,Donald Trump Jr. Pitches ‘Patriotic Capitalism...,"His firm, 1789 Capital, has been investing in ...",0.356638


## Save artifacts for Streamlit

The mapping contains the display fields needed by recommendation cards. The fitted neighbor model retains the sparse TF-IDF matrix internally, so the app never rebuilds NLP or cosine similarity.

In [21]:
mapping_columns = [
    'article_id', 'category', 'headline', 'abstract', 'snippet',
    'lead_paragraph', 'keywords', 'section', 'publication_date',
    'byline', 'url', 'image_url',
]
for column in mapping_columns:
    if column not in df:
        df[column] = None
mapping_frame = df[mapping_columns].astype(object)
mapping_frame = mapping_frame.where(pd.notna(mapping_frame), None)
news_mapping = mapping_frame.to_dict(orient='records')
joblib.dump(vectorizer, MODELS_PATH / 'tfidf_vectorizer.pkl')
joblib.dump({'tfidf_matrix': tfidf_matrix}, MODELS_PATH / 'cosine_similarity.pkl')
joblib.dump(news_mapping, MODELS_PATH / 'news_mapping.pkl')
print(f'Saved recommendation artifacts to {MODELS_PATH}')

Saved recommendation artifacts to /home/sahil/Desktop/AI-ML/Projects/Ideas/News Recommendation System - Ref/models
